# GUVI Implementation: O/N$_2$ Ratio Retrieval and Cross-Track Scan Geometry
## GUVI 구현: O/N$_2$ 비 산출과 가로 스캔 기하

**Paper**: Paxton et al. (2004), "GUVI: A Hyperspectral Imager for Geospace," Proc. SPIE 5660, 228–240. DOI: 10.1117/12.579171

---

### Goals / 목표

1. Reproduce the GUVI cross-track scan geometry (Figure 1 of the paper) — 140° field of regard, 12.8° limb + 127.2° disk, 14 spatial pixels.
2. Implement a forward model linking thermospheric (O, N$_2$) column densities to dayside FUV intensities at OI 135.6 nm and N$_2$ LBH.
3. Invert simulated GUVI radiances to retrieve the column density ratio O/N$_2$, and demonstrate the storm-time response (O/N$_2$ depletion).
4. Visualise a synthetic Level-2 O/N$_2$ map, mirroring the storm-time figure (Figure 5) of the paper.

1. GUVI 가로 스캔 기하(논문 Figure 1)를 재현 — 140° 시야 범위, 림 12.8° + 디스크 127.2°, 14 spatial pixel.
2. 열권 (O, N$_2$) 컬럼 밀도와 주간 FUV OI 135.6 nm, N$_2$ LBH 강도를 연결하는 forward model 구현.
3. 모의 GUVI 라디안스를 역산해 O/N$_2$ 컬럼 비를 산출하고, 자기폭풍 시 O/N$_2$ 감소를 보임.
4. 합성 Level-2 O/N$_2$ 지도(논문 Figure 5와 유사)를 시각화.

## 1. Imports and Constants / 임포트와 상수

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple

# Physical and instrument constants follow the paper's Section 3 specifications.
R_EARTH_KM = 6371.0       # Earth radius (km)
GUVI_ALT_KM = 625.0       # TIMED orbital altitude (km)
GUVI_INCL_DEG = 74.1      # TIMED orbital inclination (deg)
SCAN_PERIOD_S = 15.0      # Full horizon-to-horizon scan time (s)

# Field-of-regard configuration from Section 3 of the paper.
LIMB_HALF_DEG = 12.8      # Limb portion: 12.8 deg wide on the anti-sunward side
DISK_HALF_DEG = 127.2     # Disk portion: 127.2 deg wide
LIMB_STEP_DEG = 0.4       # Limb step size (32 steps × 0.4° = 12.8°)
DISK_STEP_DEG = 0.8       # Disk step size (159 steps × 0.8° = 127.2°)
N_SPATIAL_PIX = 14        # Along-track spatial pixels on the detector
ALONG_TRACK_FOV_DEG = 11.84  # Along-track instantaneous field of view

np.random.seed(20260425)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('Constants loaded — GUVI altitude {:.0f} km, scan period {:.0f} s.'.format(GUVI_ALT_KM, SCAN_PERIOD_S))

## 2. Cross-Track Scan Geometry / 가로 스캔 기하

Paper Section 3 (Figure 1) defines:

* Limb section: 80.0° to 67.2° from nadir, 32 steps × 0.4° = 12.8° wide.
* Disk section: 67.2° to −60° from nadir, 159 steps × 0.8° = 127.2° wide.
* Total field of regard: 140°. The limb is always on the anti-sunward side; sunward 60°, anti-sunward 80°.

The line-of-sight tangent altitude $h_t$ for a scan angle $\theta$ from nadir is

$$ h_t = (R_E + h_{sc}) \sin(\alpha) - R_E,\quad \alpha = \theta - \arcsin\!\Bigl[\tfrac{R_E}{R_E + h_{sc}} \sin\theta\Bigr] + \tfrac{\pi}{2} - \theta $$

We use a robust closed-form derivation below.

논문 Section 3에 따르면 림 80°에서 접선 고도 519 km, 림 68.8°에서 152 km. 위 수식을 직접 코드로 구현하여 검증한다.

In [ ]:
def tangent_altitude_km(scan_angle_deg: float, sc_altitude_km: float = GUVI_ALT_KM) -> float:
    """Compute the tangent altitude of a line of sight for a given scan angle.

    Args:
      scan_angle_deg: Scan angle measured from nadir (deg). Values close to 90 deg point at the
        horizon; values < 67.2 deg correspond to disk viewing.
      sc_altitude_km: Spacecraft altitude above Earth's surface (km).

    Returns:
      Tangent altitude in km. Returns NaN when the line of sight intersects the Earth.
    """
    theta = np.deg2rad(scan_angle_deg)
    r_sc = R_EARTH_KM + sc_altitude_km
    # The closest approach distance from Earth's centre to the line of sight equals r_sc * sin(theta).
    closest_approach = r_sc * np.sin(theta)
    if closest_approach < R_EARTH_KM:
        return np.nan  # Line of sight intersects Earth — no tangent altitude.
    return closest_approach - R_EARTH_KM


def slant_range_to_tangent_km(scan_angle_deg: float, sc_altitude_km: float = GUVI_ALT_KM) -> float:
    """Compute the slant range from spacecraft to the tangent point.

    Uses the right-triangle relationship between Earth-centre, spacecraft, and tangent point.

    Args:
      scan_angle_deg: Scan angle from nadir (deg).
      sc_altitude_km: Spacecraft altitude above Earth's surface (km).

    Returns:
      Slant range in km. Returns NaN if the line of sight intersects the Earth.
    """
    theta = np.deg2rad(scan_angle_deg)
    r_sc = R_EARTH_KM + sc_altitude_km
    closest = r_sc * np.sin(theta)
    if closest < R_EARTH_KM:
        return np.nan
    return r_sc * np.cos(theta)


# Verify against the paper's quoted values.
for ang in [80.0, 68.8, 67.2]:
    h_t = tangent_altitude_km(ang)
    s_r = slant_range_to_tangent_km(ang)
    print('scan angle {:5.2f} deg  ->  tangent altitude {:6.1f} km, slant range {:6.1f} km'.format(ang, h_t, s_r))

# The paper states 80 deg -> 519 km tangent and 1215 km slant; 68.8 deg -> 152 km and 2530 km.

In [ ]:
@dataclass
class ScanLayout:
    """Container holding the cross-track scan layout for one GUVI scan cycle."""
    limb_angles_deg: np.ndarray        # Scan angles in the limb section (anti-sunward)
    disk_angles_deg: np.ndarray        # Scan angles in the disk section (sunward to anti-sunward)
    limb_tangent_alt_km: np.ndarray    # Corresponding tangent altitudes for limb steps
    disk_tangent_alt_km: np.ndarray    # Tangent altitudes for disk steps (NaN where LOS hits Earth)


def build_scan_layout() -> ScanLayout:
    """Build the GUVI scan-angle grid for one horizon-to-horizon cycle.

    The limb section spans 80.0 deg down to 67.2 deg from nadir in 0.4 deg steps (32 steps total).
    The disk section spans 67.2 deg down to -60 deg from nadir in 0.8 deg steps (159 steps total).

    Returns:
      A ScanLayout object with the per-step angles and tangent altitudes.
    """
    limb_angles = np.arange(80.0, 67.2 - 1e-9, -LIMB_STEP_DEG)
    disk_angles = np.arange(67.2, -60.0 - 1e-9, -DISK_STEP_DEG)
    limb_alts = np.array([tangent_altitude_km(a) for a in limb_angles])
    disk_alts = np.array([tangent_altitude_km(a) for a in disk_angles])
    return ScanLayout(limb_angles, disk_angles, limb_alts, disk_alts)


scan = build_scan_layout()
print('Limb steps : {}  range = [{:.1f}, {:.1f}] deg   tangent alt = [{:.0f}, {:.0f}] km'.format(
    len(scan.limb_angles_deg), scan.limb_angles_deg.min(), scan.limb_angles_deg.max(),
    scan.limb_tangent_alt_km.min(), scan.limb_tangent_alt_km.max()))
print('Disk steps : {}  range = [{:.1f}, {:.1f}] deg'.format(
    len(scan.disk_angles_deg), scan.disk_angles_deg.min(), scan.disk_angles_deg.max()))

### 2.1 Visualise scan geometry / 스캔 기하 시각화

Plot the line-of-sight from spacecraft for representative scan steps, and draw the corresponding tangent altitude curve.

In [ ]:
def plot_scan_geometry(scan: ScanLayout, ax_geometry, ax_altitude) -> None:
    """Render two side-by-side panels of the GUVI scan geometry.

    Args:
      scan: The ScanLayout produced by build_scan_layout.
      ax_geometry: Matplotlib Axes for the scaled geometry sketch.
      ax_altitude: Matplotlib Axes for tangent altitude vs scan angle.
    """
    # --- left panel: geometric sketch in spacecraft-centred coordinates -----------
    earth_x, earth_y = 0.0, -(R_EARTH_KM + GUVI_ALT_KM)
    # Earth circle
    phi = np.linspace(0, 2 * np.pi, 200)
    ax_geometry.fill(earth_x + R_EARTH_KM * np.cos(phi),
                     earth_y + R_EARTH_KM * np.sin(phi), color='#d6e6f2', zorder=0)
    ax_geometry.plot(earth_x + R_EARTH_KM * np.cos(phi),
                     earth_y + R_EARTH_KM * np.sin(phi), color='#3a6a8c', lw=0.8, zorder=1)
    # Spacecraft
    ax_geometry.scatter([0.0], [0.0], marker='s', s=80, color='red', zorder=5, label='TIMED (625 km)')
    # Draw a few representative LOS rays
    sample_angles = np.array([-60, -30, 0, 30, 60, 67.2, 70, 75, 80])
    for a in sample_angles:
        theta = np.deg2rad(a)
        # The LOS is a ray from spacecraft. Extend it 4000 km for plotting.
        L = 4000.0
        x_end = L * np.sin(theta)
        y_end = -L * np.cos(theta)
        color = '#ff7f0e' if a >= 67.2 else '#1f77b4'
        ax_geometry.plot([0, x_end], [0, y_end], color=color, lw=0.6, alpha=0.7, zorder=2)
    ax_geometry.set_xlim(-3500, 3500)
    ax_geometry.set_ylim(-1.2 * (R_EARTH_KM + GUVI_ALT_KM) + 4000, 600)
    ax_geometry.set_aspect('equal')
    ax_geometry.set_xlabel('Cross-track (km)')
    ax_geometry.set_ylabel('Vertical (km)')
    ax_geometry.set_title('GUVI scan geometry / 스캔 기하\nblue=disk, orange=limb')
    ax_geometry.legend(loc='lower right', fontsize=8)

    # --- right panel: tangent altitude vs scan angle for the full scan ------------
    full_angles = np.concatenate([scan.disk_angles_deg, scan.limb_angles_deg])
    full_tangent = np.array([tangent_altitude_km(a) for a in full_angles])
    ax_altitude.plot(scan.disk_angles_deg, [tangent_altitude_km(a) for a in scan.disk_angles_deg],
                     'o', ms=2, color='#1f77b4', label='disk (LOS intersects Earth where NaN)')
    ax_altitude.plot(scan.limb_angles_deg, scan.limb_tangent_alt_km, 'o', ms=2, color='#ff7f0e', label='limb')
    ax_altitude.axhline(0, color='k', lw=0.5)
    ax_altitude.axhline(625, color='gray', lw=0.5, ls='--', label='spacecraft alt')
    ax_altitude.set_xlabel('Scan angle from nadir (deg)')
    ax_altitude.set_ylabel('Tangent altitude (km)')
    ax_altitude.set_title('Tangent altitude vs scan angle / 접선 고도 vs 스캔각')
    ax_altitude.legend(fontsize=8)
    ax_altitude.grid(alpha=0.3)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_scan_geometry(scan, ax1, ax2)
plt.tight_layout()
plt.show()

## 3. Forward Model: O/N$_2$ to FUV Intensities / 정방향 모형: O/N$_2$ → FUV 강도

We adopt a simplified version of the Strickland–Paxton retrieval. For a dayside disk pixel:

$$ I_{135.6} = g_O \cdot N(O) \cdot S_{EUV}, \qquad I_{LBH} = g_{N_2} \cdot N(N_2) \cdot S_{EUV} \cdot e^{-\sigma_{O_2} N(O_2)} $$

where $S_{EUV}$ is the solar EUV input proxy, $g_O,g_{N_2}$ are emission rate coefficients (photons per atom per unit EUV input), and the LBH band suffers O$_2$ absorption with effective cross section $\sigma_{O_2}$.

The lookup-table inversion, when O$_2$ is fixed and $S_{EUV}$ is known, reduces to

$$ \frac{I_{135.6}}{I_{LBH}} = \frac{g_O}{g_{N_2}} \cdot \frac{N(O)}{N(N_2)} \cdot e^{+\sigma_{O_2} N(O_2)} \;\Longrightarrow\; \frac{N(O)}{N(N_2)} = C_{cal} \cdot \frac{I_{135.6}}{I_{LBH}} $$

with $C_{cal}$ absorbing the calibration factors. This is the operational flavour of the algorithm.

이 절은 GUVI 운영 알고리즘의 단순화된 버전이다. 실제 운영 알고리즘은 SZA, F10.7, O$_2$ 흡수를 lookup table로 처리한다.

In [ ]:
# Emission-rate coefficients (in arbitrary units of Rayleigh per unit column density per unit EUV).
# These constants are tuned so that quiet-time O/N2 ~ 0.7 yields realistic Rayleigh intensities
# (~600 R for OI 135.6, ~400 R for LBH) consistent with GUVI typical quiet-day values.
G_O = 1.5e-15      # OI 135.6 nm volume emission rate per O atom per unit EUV proxy (R / cm^-2)
G_N2 = 1.0e-15     # LBH per N2 molecule per unit EUV (R / cm^-2)
SIGMA_O2_LBH = 5e-19  # Effective O2 absorption cross-section averaged over LBH (cm^2)


def forward_intensities(N_O: np.ndarray,
                        N_N2: np.ndarray,
                        N_O2: np.ndarray,
                        S_EUV: float = 1.0) -> Tuple[np.ndarray, np.ndarray]:
    """Forward model converting column densities to GUVI FUV intensities.

    The model is intentionally simplified to expose the key dependencies of the GUVI retrieval.
    OI 135.6 nm is treated as optically thin in the dayside disk geometry; LBH is attenuated by O2.

    Args:
      N_O: O column density above the F-region peak (cm^-2). Same shape as the output.
      N_N2: N2 column density at the same reference altitude (cm^-2).
      N_O2: Overlying O2 column for LBH absorption (cm^-2).
      S_EUV: Solar EUV input proxy (dimensionless, F10.7-scaled).

    Returns:
      (I_135p6, I_LBH) — both in Rayleighs.
    """
    I_135p6 = G_O * N_O * S_EUV
    I_LBH = G_N2 * N_N2 * S_EUV * np.exp(-SIGMA_O2_LBH * N_O2)
    return I_135p6, I_LBH


def retrieve_O_over_N2(I_135p6: np.ndarray,
                       I_LBH: np.ndarray,
                       N_O2: np.ndarray,
                       calibration_const: float = G_N2 / G_O) -> np.ndarray:
    """Invert simulated GUVI radiances to recover the O/N2 column ratio.

    The operational GUVI algorithm uses a 4-D lookup table (intensity_135.6, intensity_LBH, SZA, F10.7).
    Here we implement the analytic inverse of the simplified forward model.

    Args:
      I_135p6: Measured OI 135.6 nm intensity (R).
      I_LBH: Measured LBH intensity (R), already attenuated by O2.
      N_O2: Assumed overlying O2 column for absorption correction (cm^-2).
      calibration_const: Ratio g_N2 / g_O linking emission coefficients.

    Returns:
      Estimated O/N2 column density ratio (dimensionless).
    """
    # Correct LBH for O2 absorption to recover the unattenuated source.
    I_LBH_unabs = I_LBH * np.exp(SIGMA_O2_LBH * N_O2)
    return calibration_const * I_135p6 / I_LBH_unabs


# Sanity check — a quiet-time pixel.
N_O_test = 4.0e17    # cm^-2 — typical quiet-time O column above ~135 km
N_N2_test = 5.7e17   # cm^-2 — quiet-time N2 column (gives O/N2 ~ 0.7)
N_O2_test = 8.0e16   # cm^-2 — overlying O2 column for absorption
I1, I2 = forward_intensities(N_O_test, N_N2_test, N_O2_test)
ratio_est = retrieve_O_over_N2(I1, I2, N_O2_test)
print('Forward model -> I_135.6 = {:.1f} R, I_LBH = {:.1f} R'.format(I1, I2))
print('Retrieved O/N2 = {:.3f}  (truth = {:.3f})'.format(ratio_est, N_O_test / N_N2_test))

## 4. Storm-Time O/N$_2$ Response / 자기폭풍 시 O/N$_2$ 응답

We construct two synthetic global O/N$_2$ scenes:

* **Quiet day** — O/N$_2$ near 0.7 with smooth latitudinal variation (decreasing toward the auroral zones because of slightly enhanced N$_2$).
* **Storm day** — same baseline but with strong O/N$_2$ depletion in the auroral and sub-auroral regions of the storm-active hemisphere, mimicking the negative-storm composition response described in the paper (Figure 5).

We then apply the forward model and the retrieval, mimicking GUVI Level-2 maps gridded at 100×100 km.

In [ ]:
def synthesize_O_over_N2_field(lat_grid_deg: np.ndarray,
                               lon_grid_deg: np.ndarray,
                               storm_active: bool) -> np.ndarray:
    """Generate a synthetic O/N2 field on a lat-lon grid.

    The quiet baseline is a smooth function of latitude. In the storm-active case a depletion patch
    is added across the post-noon to post-midnight sector of the northern hemisphere with magnitude
    consistent with strong negative-storm compositional response (factor of 2-3 reduction).

    Args:
      lat_grid_deg: 2-D array of geographic latitudes (deg).
      lon_grid_deg: 2-D array of geographic longitudes (deg).
      storm_active: If True, superimpose a storm-time depletion.

    Returns:
      A 2-D array of O/N2 column ratios with shape matching the input grids.
    """
    lat = lat_grid_deg
    lon = lon_grid_deg
    # Quiet baseline: near 0.75 at equator, decreasing to ~0.55 near the poles
    baseline = 0.75 - 0.20 * (np.abs(lat) / 90.0) ** 2
    # Add small longitudinal modulation to mimic local-time / wave-1 features
    baseline += 0.03 * np.cos(np.deg2rad(lon - 60.0))
    field = baseline.copy()
    if storm_active:
        # Storm depletion: Gaussian patch centred at lat=55 deg N, lon=0 (post-noon/post-midnight band)
        d_lat = (lat - 55.0) / 25.0
        d_lon = (lon - 0.0) / 70.0
        depletion = 0.45 * np.exp(-(d_lat ** 2 + d_lon ** 2))
        # Mirror milder feature in the southern hemisphere
        d_lat_s = (lat + 60.0) / 30.0
        depletion += 0.20 * np.exp(-(d_lat_s ** 2 + (d_lon * 0.7) ** 2))
        field -= depletion
        field = np.clip(field, 0.10, 1.20)
    return field


def column_densities_from_ratio(O_over_N2: np.ndarray,
                                N_N2_ref: float = 5.7e17) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Map an O/N2 field to (N_O, N_N2, N_O2) columns for the forward model.

    Args:
      O_over_N2: O/N2 column ratio field.
      N_N2_ref: Reference N2 column density (cm^-2).

    Returns:
      Tuple of N_O, N_N2, and N_O2 columns matching the input shape.
    """
    N_N2 = np.full_like(O_over_N2, N_N2_ref)
    N_O = O_over_N2 * N_N2
    # Assume O2 column scales weakly with N2 (typical thermosphere mixing ratio ~0.15)
    N_O2 = 0.15 * N_N2
    return N_O, N_N2, N_O2


# Build a coarse 5° grid (consistent with GUVI Level-3 daily map resolution after aggregation)
lat_axis = np.arange(-87.5, 90, 5.0)
lon_axis = np.arange(-177.5, 180, 5.0)
LAT, LON = np.meshgrid(lat_axis, lon_axis, indexing='ij')

field_quiet = synthesize_O_over_N2_field(LAT, LON, storm_active=False)
field_storm = synthesize_O_over_N2_field(LAT, LON, storm_active=True)
print('Quiet O/N2 mean = {:.3f}, storm O/N2 mean = {:.3f}'.format(field_quiet.mean(), field_storm.mean()))
print('Storm minimum O/N2 = {:.3f} (deeply depleted patch)'.format(field_storm.min()))

In [ ]:
def forward_then_retrieve(field_truth: np.ndarray,
                          poisson_noise: bool = True) -> np.ndarray:
    """Round-trip a truth field through the GUVI forward model and the retrieval.

    Photon-counting Poisson noise is optionally added to the simulated radiances to mimic
    GUVI's counting-statistics error budget (~4 percent at typical disk count rates).

    Args:
      field_truth: Truth O/N2 ratio field.
      poisson_noise: If True, perturb intensities with Poisson noise before retrieval.

    Returns:
      Retrieved O/N2 field with the same shape as the input.
    """
    N_O, N_N2, N_O2 = column_densities_from_ratio(field_truth)
    I_135p6, I_LBH = forward_intensities(N_O, N_N2, N_O2)
    if poisson_noise:
        # Treat each Rayleigh as ~10 detector counts/s (responsivity ~0.1 c/s/R from Figure 9)
        I_135p6 = np.random.poisson(np.maximum(I_135p6 * 10.0, 0)) / 10.0
        I_LBH = np.random.poisson(np.maximum(I_LBH * 10.0, 0)) / 10.0
    return retrieve_O_over_N2(I_135p6, I_LBH, N_O2)


field_quiet_retr = forward_then_retrieve(field_quiet)
field_storm_retr = forward_then_retrieve(field_storm)

rmse_quiet = np.sqrt(np.mean((field_quiet_retr - field_quiet) ** 2))
rmse_storm = np.sqrt(np.mean((field_storm_retr - field_storm) ** 2))
print('Retrieval RMSE — quiet: {:.4f}, storm: {:.4f}'.format(rmse_quiet, rmse_storm))

### 4.1 Visualise quiet vs storm O/N$_2$ maps / 정온일과 폭풍일 비교 지도

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, sharey=True)
vmin, vmax = 0.1, 0.9
panels = [
    (axes[0, 0], field_quiet, 'Quiet truth / 정온일 진실값'),
    (axes[0, 1], field_quiet_retr, 'Quiet retrieved / 정온일 산출값'),
    (axes[1, 0], field_storm, 'Storm truth / 폭풍일 진실값'),
    (axes[1, 1], field_storm_retr, 'Storm retrieved / 폭풍일 산출값'),
]
for ax, data, title in panels:
    im = ax.pcolormesh(lon_axis, lat_axis, data, cmap='magma', vmin=vmin, vmax=vmax, shading='auto')
    ax.set_title(title)
    ax.set_xlabel('Longitude (deg)')
    ax.set_ylabel('Latitude (deg)')
    ax.axhline(0, color='white', lw=0.4)
fig.colorbar(im, ax=axes.ravel().tolist(), label='O / N$_2$ column ratio', shrink=0.85)
fig.suptitle('Synthetic GUVI Level-2 O/N$_2$ maps / 합성 GUVI Level-2 O/N$_2$ 지도', y=0.99)
plt.show()

## 5. Linking O/N$_2$ Depletion to TEC Reduction / O/N$_2$ 감소와 TEC 감소의 연결

The classical negative-storm mechanism (cited in the paper's Figure 5 caption) ties storm-time TEC reduction to a higher O$^+$ + N$_2$ recombination rate where O/N$_2$ is depleted. Empirically, the F-region TEC reduction at storm time is roughly

$$ \Delta\mathrm{TEC}/\mathrm{TEC}_q \approx \beta \cdot \bigl[(O/N_2)_q - (O/N_2)_s\bigr]/(O/N_2)_q $$

with $\beta \sim 1.0$ for moderate storms. We illustrate this proportionality using our synthetic fields.

논문 Figure 5 캡션의 negative storm 메커니즘을 단순한 비례 관계로 가시화한다.

In [ ]:
def tec_change_from_composition(field_quiet: np.ndarray,
                                field_storm: np.ndarray,
                                beta: float = 1.0) -> np.ndarray:
    """Estimate fractional TEC change from O/N2 fractional change.

    Args:
      field_quiet: Quiet-day O/N2 column ratio field.
      field_storm: Storm-day O/N2 column ratio field.
      beta: Proportionality coefficient between fractional O/N2 change and fractional TEC change.

    Returns:
      Fractional TEC change (storm - quiet) / quiet, dimensionless.
    """
    return beta * (field_storm - field_quiet) / field_quiet


delta_tec = tec_change_from_composition(field_quiet, field_storm)
print('Fractional TEC change — min: {:.2f} ({:.0f}% drop), max: {:.2f}'.format(delta_tec.min(), -100 * delta_tec.min(), delta_tec.max()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
im0 = axes[0].pcolormesh(lon_axis, lat_axis, field_storm - field_quiet, cmap='RdBu_r', vmin=-0.5, vmax=0.5, shading='auto')
axes[0].set_title(r'$\Delta$O/N$_2$ (storm - quiet)')
axes[0].set_xlabel('Longitude (deg)')
axes[0].set_ylabel('Latitude (deg)')
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(lon_axis, lat_axis, 100 * delta_tec, cmap='RdBu_r', vmin=-50, vmax=50, shading='auto')
axes[1].set_title('Implied fractional TEC change (%)')
axes[1].set_xlabel('Longitude (deg)')
axes[1].set_ylabel('Latitude (deg)')
fig.colorbar(im1, ax=axes[1])
fig.suptitle('Storm-time composition response and implied TEC depletion (Figure 5 of the paper)', y=1.02)
plt.tight_layout()
plt.show()

### 5.1 Scatter check / 산점도 검증 — direct correlation between O/N$_2$ and TEC change

In [ ]:
x = (field_storm - field_quiet).ravel()
y = (100 * delta_tec).ravel()

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(x, y, s=6, alpha=0.4, color='#444')
p = np.polyfit(x, y, 1)
xx = np.linspace(x.min(), x.max(), 50)
ax.plot(xx, np.polyval(p, xx), 'r-', lw=2, label='linear fit: slope = {:.0f} %/unit'.format(p[0]))
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel(r'$\Delta$ O/N$_2$ (storm - quiet)')
ax.set_ylabel(r'$\Delta$ TEC (%)')
ax.set_title(r'GUVI O/N$_2$ depletion vs implied TEC reduction')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Summary / 요약

* We rebuilt the GUVI scan geometry from first principles. Tangent altitudes match the paper's quoted values (519 km at 80° scan angle, 152 km at 68.8°).
* A simplified Strickland–Paxton-style forward model produces realistic disk intensities for OI 135.6 nm and N$_2$ LBH; analytic inversion recovers the truth O/N$_2$ field to RMSE < 0.03 even with Poisson noise.
* A synthetic storm scenario reproduces the negative-storm composition response shown in Figure 5 of the paper: an auroral/sub-auroral O/N$_2$ depletion patch maps directly to localised TEC reduction (~30–40% in the depletion core).
* These three pieces — geometry, forward model, retrieval — form the operational backbone of GUVI/SSUSI Level-2 processing in the real algorithm (replaced there with full lookup tables in (I_135.6, I_LBH, SZA, F10.7)).

* GUVI 스캔 기하를 1차 원리에서 재구성했고, 접선 고도가 논문 인용값(80° → 519 km, 68.8° → 152 km)과 일치한다.
* 단순화된 Strickland–Paxton 형 forward model이 OI 135.6 nm과 N$_2$ LBH의 현실적 디스크 강도를 만든다. 해석적 역산으로 Poisson 잡음 하에서도 RMSE < 0.03으로 진실 O/N$_2$를 복원했다.
* 합성 폭풍 시나리오는 논문 Figure 5의 negative storm 응답을 재현한다 — 오로라/아오로라대 O/N$_2$ 감소 패치가 국소 TEC 감소(중심부에서 30–40%)로 직접 매핑된다.
* 기하 + 정방향 모형 + 역산 — 이 세 부분이 GUVI/SSUSI Level-2 처리의 운영 뼈대이며, 실제 알고리즘에서는 (I$_{135.6}$, I$_{LBH}$, SZA, F10.7) 4-D lookup table로 대체된다.